# Text Preprocessing Pipeline
A professional NLP preprocessing pipeline for social media / Steam game reviews.

## 1. Imports & Setup

In [34]:
!pip install emoji googletrans==4.0.0rc1


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import pandas as pd
import re
from textblob import Word, TextBlob
from symspellpy.symspellpy import SymSpell, Verbosity
from tqdm.notebook import tqdm
import emoji
from googletrans import Translator
from nltk.corpus import stopwords
import nltk
import pkg_resources

nltk.download('stopwords', quiet=True)
# tqdm.pandas()

translator = Translator()

# Initialize SymSpell
sym_spell = SymSpell(max_dictionary_edit_distance=2)
dictionary_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dictionary_path, 0, 1)

print("Setup complete.")

Setup complete.


## 2. Cleaning Functions

In [36]:
def translate_to_english(text):
    if pd.isna(text) or str(text).strip() == "":   # <-- use pd.isna, not == 'nan'
        return text
    try:
        result = translator.translate(text, dest='en')
        return result.text
    except Exception:
        return text

def fix_encoding_fn(text):
    if pd.isna(text):
        return text
    return text.encode("ascii", "ignore").decode()

def remove_noise_fn(text):
    if pd.isna(text):
        return text
    text = re.sub(r"\.{2,}", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text.strip()

def lowercase_text(text):
    if pd.isna(text):
        return text
    return text.lower()

def remove_numbers_fn(text):
    if pd.isna(text):
        return text
    return re.sub(r"\d+", "", text)

def fix_spelling_fn(text):   # <-- was: def fix_spelling(text)
    if pd.isna(text):
        return text
    words = text.split()
    corrected = []
    for word in words:
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        if suggestions:
            corrected.append(suggestions[0].term)
        else:
            corrected.append(str(TextBlob(word).correct()))
    return " ".join(corrected)

def lemmatize_text(text):
    if pd.isna(text):
        return text
    words = text.split()
    lemmas = [Word(w).lemmatize() for w in words]
    return " ".join(lemmas)

def remove_stopwords_fn(text):
    if pd.isna(text):
        return text
    opinion_words = {
        'no', 'not', 'nor', 'never', 'neither', 'nobody', 'nothing',
        'nowhere', 'hardly', 'scarcely', 'barely', "don't", "doesn't",
        "didn't", "won't", "wouldn't", "shouldn't", "couldn't", "isn't",
        "aren't", "wasn't", "weren't"
    }
    stop_words = set(stopwords.words('english')) - opinion_words
    words = text.split()
    return " ".join([w for w in words if w.lower() not in stop_words])

def remove_emojis_fn(text):
    if pd.isna(text):
        return text
    return emoji.replace_emoji(text, replace='').strip()

def extract_genre(text):
    if pd.isna(text):
        return "Unknown"
    match = re.search(r"'(.*?)'", text)
    return match.group(1) if match else "Unknown"

print("Functions defined.")

Functions defined.


## 3. Pipeline Function

Each keyword argument maps directly to one of the original CLI flags.

In [ ]:
def run_pipeline(
    input_file,
    output_file,
    text_column="review_text",
    category_column="genres",
    translate=False,
    fix_encoding=False,
    remove_noise=False,
    lowercase=False,
    remove_numbers=False,
    fix_spelling=False,
    lemmatize=False,
    extract_tags=False,
    remove_stopwords=False,
    remove_emojis=False,
):
    """
    Run the text preprocessing pipeline on a CSV file.
    Each flag corresponds to one CLI flag from the original script.
    Returns the cleaned DataFrame and saves it to output_file.
    """
    print(f"\n{'='*60}")
    print(f"  INPUT : {input_file}")
    print(f"  OUTPUT: {output_file}")
    print(f"{'='*60}")

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df):,} rows.\n")

    # Convert text column to object to avoid pyarrow issues
    df[text_column] = df[text_column].astype(object)

    # Ordered steps: (flag, label, function)
    steps = [
    (translate,        "Translating to English...",    translate_to_english),
    (fix_encoding,     "Fixing encoding artifacts...", fix_encoding_fn),
    (remove_noise,     "Removing noise...",            remove_noise_fn),
    (lowercase,        "Converting to lowercase...",   lowercase_text),
    (remove_numbers,   "Removing numbers...",          remove_numbers_fn),
    (fix_spelling,     "Fixing spelling...",           fix_spelling_fn),   # <-- _fn added
    (lemmatize,        "Applying lemmatization...",    lemmatize_text),
    (remove_stopwords, "Removing stopwords...",        remove_stopwords_fn),
    (remove_emojis,    "Removing emojis...",           remove_emojis_fn),
    ]

    for flag, label, fn in steps:
        if flag:
            print(label)
            df[text_column] = df[text_column].apply(fn)

    if extract_tags:
        print("Extracting category tags...")
        df['category'] = df[category_column].apply(extract_genre)

    print(f"\n--- Summary ---")
    print(f"Rows             : {len(df):,}")
    print(f"Empty text rows  : {df[text_column].isna().sum()}")
    if extract_tags and 'category' in df.columns:
        print(f"Unique categories: {df['category'].nunique()}")

    df.to_csv(output_file, index=False)
    print(f"\nSaved -> {output_file}")

    return df

print("run_pipeline() ready.")

run_pipeline() ready.


## 4. Produce Datasets

In [38]:
df = pd.read_csv("../data/GROUBD_TRUTH_WITH_FINAL_LABEL.CSV")

# Remove the Last 4 columns
df = df.iloc[:, :-4]

# Save the cleaned DataFrame to a new CSV file
df.to_csv("STEAM_GAMES_REDUCED.CSV", index=False)

1. Full pipeline on ground-truth labels → `STEAM_GAMES_CLEAN_GTLabel.csv`

In [39]:
df_gt = run_pipeline(
    input_file       = "../data/GROUBD_TRUTH_WITH_FINAL_LABEL.CSV",
    output_file      = "STEAM_GAMES_CLEAN_GTLabel.csv",
    translate        = True,
    fix_encoding     = True,
    remove_noise     = True,
    lowercase        = True,
    remove_numbers   = True,
    fix_spelling     = True,
    lemmatize        = True,
    extract_tags     = True,
    remove_stopwords = True,
    remove_emojis    = True,
)
df_gt.head()


  INPUT : ../data/GROUBD_TRUTH_WITH_FINAL_LABEL.CSV
  OUTPUT: STEAM_GAMES_CLEAN_GTLabel.csv
Loaded 200 rows.

Translating to English...
Fixing encoding artifacts...
Removing noise...
Converting to lowercase...
Removing numbers...
Fixing spelling...
Applying lemmatization...
Removing stopwords...
Removing emojis...
Extracting category tags...

--- Summary ---
Rows             : 200
Empty text rows  : 0
Unique categories: 2

Saved -> STEAM_GAMES_CLEAN_GTLabel.csv


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,...,genres,platforms,categories,release_date,price,label_1,label_2,label_3,final_label,category
0,0,1860,1860,Path of Exile 2,2694490,,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",...,"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral,neutral,neutral,neutral,Action
1,1,353,353,Palworld,1623730,incredibly fun great repeatability think spell...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",...,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive,positive,positive,positive,Action
2,2,1333,1333,Monster Hunter Wilds,2246340,monster hunter till wild,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000",...,"['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,negative,neutral,neutral,neutral,Action
3,3,905,905,Left 4 Dead 2,550,really good game love zombie apocalypse game g...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",...,['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive,positive,positive,positive,Action
4,4,1289,1289,War Thunder,236390,love game piss unlike game,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",...,"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,negative,positive,negative,negative,Action


2. Initial cleaning on raw data → `SCHEMA_1.csv`

In [40]:
df_schema1 = run_pipeline(
    input_file   = "STEAM_GAMES_REDUCED.csv",
    output_file  = "SCHEMA_1.csv",
    translate    = True,
    fix_encoding = True,
    remove_noise = True,
    lowercase    = True,
)
df_schema1.head()


  INPUT : STEAM_GAMES_REDUCED.csv
  OUTPUT: SCHEMA_1.csv
Loaded 200 rows.

Translating to English...
Fixing encoding artifacts...
Removing noise...
Converting to lowercase...

--- Summary ---
Rows             : 200
Empty text rows  : 0

Saved -> SCHEMA_1.csv


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,1860,1860,Path of Exile 2,2694490,,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0
1,1,353,353,Palworld,1623730,incredibly fun and great replayability i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0
2,2,1333,1333,Monster Hunter Wilds,2246340,i monster my hunter till i wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0
3,3,905,905,Left 4 Dead 2,550,this is a really good game if you love zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0
4,4,1289,1289,War Thunder,236390,love the game but it pisses me the off unlike...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN


3. Numbers, emojis & lemmatization on Schema 1 → `SCHEMA_2.csv`

In [41]:
df_schema2 = run_pipeline(
    input_file     = "SCHEMA_1.csv",
    output_file    = "SCHEMA_2.csv",
    remove_numbers = True,
    remove_emojis  = True,
    lemmatize      = True,
)
df_schema2.head()


  INPUT : SCHEMA_1.csv
  OUTPUT: SCHEMA_2.csv
Loaded 200 rows.

Removing numbers...
Applying lemmatization...
Removing emojis...

--- Summary ---
Rows             : 200
Empty text rows  : 6

Saved -> SCHEMA_2.csv


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,1860,1860,Path of Exile 2,2694490,NaN,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0
1,1,353,353,Palworld,1623730,incredibly fun and great replayability i think...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0
2,2,1333,1333,Monster Hunter Wilds,2246340,i monster my hunter till i wild,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0
3,3,905,905,Left 4 Dead 2,550,this is a really good game if you love zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0
4,4,1289,1289,War Thunder,236390,love the game but it piss me the off unlike an...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN


4. Spelling & stopwords on Schema 2 → `SCHEMA_3.csv`

In [42]:
df_schema3 = run_pipeline(
    input_file       = "SCHEMA_2.csv",
    output_file      = "SCHEMA_3.csv",
    fix_spelling     = True,
    remove_stopwords = True,
)
df_schema3.head()


  INPUT : SCHEMA_2.csv
  OUTPUT: SCHEMA_3.csv
Loaded 200 rows.

Fixing spelling...
Removing stopwords...

--- Summary ---
Rows             : 200
Empty text rows  : 6

Saved -> SCHEMA_3.csv


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,1860,1860,Path of Exile 2,2694490,NaN,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0
1,1,353,353,Palworld,1623730,incredibly fun great repeatability think spell...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0
2,2,1333,1333,Monster Hunter Wilds,2246340,monster hunter till wild,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0
3,3,905,905,Left 4 Dead 2,550,really good game love zombie apocalypse game g...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0
4,4,1289,1289,War Thunder,236390,love game piss unlike game,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN
